In [32]:
import os
from pathlib import Path

import librosa
import numpy as np
import pandas as pd
from IPython.display import Audio


import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset

In [3]:
root_path = Path(r"C:\Users\bamilosin\Documents\dataset\audio\infant-cry dataset")

cry_path = root_path / "cry"
not_cry_path = root_path / "not_cry"
data_csv = root_path / "data.csv"

In [ ]:
class CryData(Dataset):
    def __init__(self, audio_path):
        super(CryData, self).__init__()
        self.data_csv = data_csv
        self.audio_path = audio_path
        self.audio_files = self.process_audio(audio_path)

    def process_audio(self, audio_path):
        audios = []
        for label in os.listdir(audio_path):
            # if is a subdir
            if os.path.isdir(audio_path / label):
                for audio_file in os.listdir(audio_path / label):
                    audios.append(audio_file)
        return audios
    
    ## to be completed!
    def audio_padder(self, audio):
        pass
        
    def __len__(self):
        return len(self.audio_files)
    
    def __getitem__(self, idx):
        audio = self.audio_files[idx]
        if 'n' in audio: 
            audio_path = self.audio_path / 'not_cry' / audio
            label = 0 
        else: 
            audio_path = self.audio_path / 'cry' / audio
            label = 1

        y, sr = librosa.load(audio_path)
        amp = librosa.stft(y) # amplitude
        db = librosa.amplitude_to_db(np.abs(amp), ref=np.max)
        
        return db, torch.tensor(label, dtype=torch.long)


In [39]:
BATCH_SIZE = 32

dataset = CryData(root_path)
dataloader = DataLoader(dataset, BATCH_SIZE, shuffle=True)

In [40]:
audio, label = next(iter(dataloader))

RuntimeError: stack expects each tensor to be equal size, but got [1025, 216] at entry 0 and [1025, 299] at entry 3